# Whole Census Tract Analysis (State-Level)

Computes state-level census tract resolution statistics on the post-tract-matcher dataset (`5_whole_census_tract_matcher_*.csv`).

Outputs:
- `output/2_analysis/tables/whole_tract_resolution_state_summary.csv`
- `output/2_analysis/figures/whole_tract_resolution_state_map.html`
- `output/2_analysis/figures/whole_tract_resolution_state_map.png`

In [1]:
import os
import re
import glob
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 200)

DATA_GLOB = '../../data/1_derived/5_whole_census_tract_matcher_0*.csv'
TABLES_DIR = '../../output/2_analysis/tables'
FIGURES_DIR = '../../output/2_analysis/figures'
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

files = sorted(glob.glob(DATA_GLOB))
if not files:
    raise FileNotFoundError(f'No files found for pattern: {DATA_GLOB}')
print(f'Found {len(files)} input files:')
for f in files:
    print('  -', os.path.basename(f))

Found 7 input files:
  - 5_whole_census_tract_matcher_001.csv
  - 5_whole_census_tract_matcher_002.csv
  - 5_whole_census_tract_matcher_003.csv
  - 5_whole_census_tract_matcher_004.csv
  - 5_whole_census_tract_matcher_005.csv
  - 5_whole_census_tract_matcher_006.csv
  - 5_whole_census_tract_matcher_007.csv


In [2]:
needed_cols = [
    'state_fips', 'full_address_std',
    'match', 'tract_resolution_method', 'final_census_tract', 'tract_resolved',
]

first_cols = pd.read_csv(files[0], nrows=0).columns.tolist()
use_cols = [c for c in needed_cols if c in first_cols]

df = pd.concat(
    [pd.read_csv(f, usecols=use_cols, low_memory=False) for f in files],
    ignore_index=True,
)
print(f'Loaded {len(df):,} rows.')
df.head()

Loaded 1,051,219 rows.


,full_address_std,match,state_fips,tract_resolution_method,final_census_tract,tract_resolved
0,"4501 CIRCLE 75 PKY SE, ATLANTA, GA 30339, UNIT...",No_Match,0.0,One-to-One Primary,0.0,True
1,"3200 DOWNWOOD CIR NW, ATLANTA, GA 30327, UNITE...",Match,13.0,One-to-One Primary,9803.0,True
2,"100 EDGEWOOD AVE NE, ATLANTA, GA 30303, UNITED...",Match,13.0,One-to-One Primary,11901.0,True
3,"1888 EMERY ST NW, ATLANTA, GA 30318, UNITED ST...",Match,13.0,One-to-One Primary,9001.0,True
4,"3500 LENOX RD NE, ATLANTA, GA 30326, UNITED ST...",Match,13.0,One-to-One Primary,9606.0,True


In [3]:
fips_to_abbrev = {
    '01': 'AL', '02': 'AK', '04': 'AZ', '05': 'AR', '06': 'CA', '08': 'CO', '09': 'CT',
    '10': 'DE', '11': 'DC', '12': 'FL', '13': 'GA', '15': 'HI', '16': 'ID', '17': 'IL',
    '18': 'IN', '19': 'IA', '20': 'KS', '21': 'KY', '22': 'LA', '23': 'ME', '24': 'MD',
    '25': 'MA', '26': 'MI', '27': 'MN', '28': 'MS', '29': 'MO', '30': 'MT', '31': 'NE',
    '32': 'NV', '33': 'NH', '34': 'NJ', '35': 'NM', '36': 'NY', '37': 'NC', '38': 'ND',
    '39': 'OH', '40': 'OK', '41': 'OR', '42': 'PA', '44': 'RI', '45': 'SC', '46': 'SD',
    '47': 'TN', '48': 'TX', '49': 'UT', '50': 'VT', '51': 'VA', '53': 'WA', '54': 'WV',
    '55': 'WI', '56': 'WY', '60': 'AS', '66': 'GU', '69': 'MP', '72': 'PR', '78': 'VI',
}

if 'state_fips' in df.columns:
    fips_clean = (
        pd.to_numeric(df['state_fips'], errors='coerce')
        .astype('Int64')
        .astype(str)
        .str.replace('<NA>', '', regex=False)
        .str.zfill(2)
    )
    state_from_fips = fips_clean.map(fips_to_abbrev)
else:
    state_from_fips = pd.Series([pd.NA] * len(df), dtype='string')

if 'full_address_std' in df.columns:
    state_from_addr = (
        df['full_address_std'].astype(str)
        .str.extract(r',\s*([A-Z]{2})\s+\d{5}(?:-\d{4})?(?:,|$)', expand=False)
    )
else:
    state_from_addr = pd.Series([pd.NA] * len(df), dtype='string')

state_abbrev = pd.Series(state_from_fips, dtype='string').combine_first(
    pd.Series(state_from_addr, dtype='string')
).str.strip().str.upper()
state_abbrev = state_abbrev.where(state_abbrev.str.match(r'^[A-Z]{2}$', na=False), pd.NA)
df['state_abbrev'] = state_abbrev

print('Rows with state derived:', int(df['state_abbrev'].notna().sum()))
print('Rows missing state:    ', int(df['state_abbrev'].isna().sum()))

Rows with state derived: 1044900
Rows missing state:     6319


In [4]:
df['_resolved'] = df['tract_resolved'].astype(bool) if 'tract_resolved' in df.columns else df['final_census_tract'].notna()
df['_ambiguous'] = df['tract_resolution_method'].eq('Tract Ambiguous')
df['_missing'] = df['tract_resolution_method'].eq('Tract Missing')
df['_method_one_to_one'] = df['tract_resolution_method'].eq('One-to-One Primary')
df['_method_zip_tie'] = df['tract_resolution_method'].eq('Tie Resolved via ZIP-Matching Ties')
df['_method_unanimous'] = df['tract_resolution_method'].eq('Tie Resolved via All Ties Unanimous')

state_summary = (
    df.dropna(subset=['state_abbrev'])
      .groupby('state_abbrev', as_index=False)
      .agg(
          total_rows=('state_abbrev', 'size'),
          resolved_rows=('_resolved', 'sum'),
          ambiguous_rows=('_ambiguous', 'sum'),
          missing_rows=('_missing', 'sum'),
          one_to_one_rows=('_method_one_to_one', 'sum'),
          tie_via_zip_rows=('_method_zip_tie', 'sum'),
          tie_unanimous_rows=('_method_unanimous', 'sum'),
      )
)
for c in ['resolved_rows', 'ambiguous_rows', 'missing_rows', 'one_to_one_rows', 'tie_via_zip_rows', 'tie_unanimous_rows']:
    state_summary[c] = pd.to_numeric(state_summary[c], errors='coerce').fillna(0).astype(int)
state_summary['resolution_rate_pct'] = (state_summary['resolved_rows'] / state_summary['total_rows'] * 100).round(2)
state_summary['ambiguous_rate_pct'] = (state_summary['ambiguous_rows'] / state_summary['total_rows'] * 100).round(2)
state_summary['missing_rate_pct'] = (state_summary['missing_rows'] / state_summary['total_rows'] * 100).round(2)
state_summary = state_summary.sort_values(['resolution_rate_pct', 'total_rows'], ascending=[False, False])

out_path = os.path.join(TABLES_DIR, 'whole_tract_resolution_state_summary.csv')
state_summary.to_csv(out_path, index=False)
print(f'Saved state summary to {out_path}')
state_summary

Saved state summary to ../../output/2_analysis/tables\whole_tract_resolution_state_summary.csv


,state_abbrev,total_rows,resolved_rows,ambiguous_rows,missing_rows,one_to_one_rows,tie_via_zip_rows,tie_unanimous_rows,resolution_rate_pct,ambiguous_rate_pct,missing_rate_pct
21,ME,2975,2973,2,0,2957,0,16,99.93,0.07,0.00
30,NH,3782,3779,3,0,3768,0,11,99.92,0.08,0.00
0,AK,1058,1057,1,0,1055,2,0,99.91,0.09,0.00
3,AZ,24670,24646,15,9,24601,43,2,99.90,0.06,0.04
16,KS,7596,7585,7,4,7560,25,0,99.86,0.09,0.05
29,NE,4953,4946,3,4,4932,13,1,99.86,0.06,0.08
50,WY,722,721,1,0,719,2,0,99.86,0.14,0.00
33,NV,11878,11860,13,5,11823,37,0,99.85,0.11,0.04
45,VA,25455,25414,31,10,25366,46,2,99.84,0.12,0.04
20,MD,18351,18320,27,4,18270,49,1,99.83,0.15,0.02


In [5]:
n_total = len(df)
n_resolved = int(df['_resolved'].sum())
n_ambiguous = int(df['_ambiguous'].sum())
n_missing = int(df['_missing'].sum())
print(f'Total rows: {n_total:,}')
print(f'Resolved:   {n_resolved:,} ({n_resolved/n_total*100:.2f}%)')
print(f'Ambiguous:  {n_ambiguous:,} ({n_ambiguous/n_total*100:.2f}%)')
print(f'Missing:    {n_missing:,} ({n_missing/n_total*100:.2f}%)')

Total rows: 1,051,219
Resolved:   1,047,897 (99.68%)
Ambiguous:  2,637 (0.25%)
Missing:    685 (0.07%)


In [6]:
# Limit map to the contiguous US + AK/HI/DC (skip territories so the legend scale is meaningful).
us_states = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','DC','FL','GA','HI','ID','IL','IN','IA',
    'KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM',
    'NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA',
    'WV','WI','WY',
}
map_df = state_summary[state_summary['state_abbrev'].isin(us_states)].copy()

fig = px.choropleth(
    map_df,
    locations='state_abbrev',
    locationmode='USA-states',
    color='resolution_rate_pct',
    hover_name='state_abbrev',
    hover_data={
        'total_rows': ':,',
        'resolved_rows': ':,',
        'ambiguous_rows': ':,',
        'missing_rows': ':,',
        'resolution_rate_pct': ':.2f',
    },
    scope='usa',
    color_continuous_scale='YlGnBu',
    range_color=(map_df['resolution_rate_pct'].min(), 100),
    title='Census Tract Resolution Rate by State (%)',
)
fig.update_layout(margin=dict(l=10, r=10, t=60, b=10))

html_path = os.path.join(FIGURES_DIR, 'whole_tract_resolution_state_map.html')
png_path = os.path.join(FIGURES_DIR, 'whole_tract_resolution_state_map.png')
fig.write_html(html_path, include_plotlyjs='cdn')
try:
    fig.write_image(png_path, width=1100, height=650, scale=2)
    print(f'Saved PNG to {png_path}')
except Exception as e:
    print(f'PNG export skipped: {e}')

print(f'Saved HTML to {html_path}')
fig.show()

Saved PNG to ../../output/2_analysis/figures\whole_tract_resolution_state_map.png
Saved HTML to ../../output/2_analysis/figures\whole_tract_resolution_state_map.html


In [7]:
min_total_for_ranking = 1000
ranked = state_summary[state_summary['total_rows'] >= min_total_for_ranking].copy()

top_states = ranked.sort_values('resolution_rate_pct', ascending=False).head(10)
bottom_states = ranked.sort_values('resolution_rate_pct', ascending=True).head(10)
biggest_states = state_summary.sort_values('total_rows', ascending=False).head(10)

top_states.to_csv(os.path.join(TABLES_DIR, 'whole_tract_resolution_top_states.csv'), index=False)
bottom_states.to_csv(os.path.join(TABLES_DIR, 'whole_tract_resolution_bottom_states.csv'), index=False)
biggest_states.to_csv(os.path.join(TABLES_DIR, 'whole_tract_resolution_biggest_states.csv'), index=False)

display(Markdown('### Top 10 states by tract resolution rate (>= 1,000 rows)'))
display(top_states)
display(Markdown('### Bottom 10 states by tract resolution rate (>= 1,000 rows)'))
display(bottom_states)
display(Markdown('### Top 10 states by row count'))
display(biggest_states)

### Top 10 states by tract resolution rate (>= 1,000 rows)

,state_abbrev,total_rows,resolved_rows,ambiguous_rows,missing_rows,one_to_one_rows,tie_via_zip_rows,tie_unanimous_rows,resolution_rate_pct,ambiguous_rate_pct,missing_rate_pct
21,ME,2975,2973,2,0,2957,0,16,99.93,0.07,0.00
30,NH,3782,3779,3,0,3768,0,11,99.92,0.08,0.00
0,AK,1058,1057,1,0,1055,2,0,99.91,0.09,0.00
3,AZ,24670,24646,15,9,24601,43,2,99.90,0.06,0.04
16,KS,7596,7585,7,4,7560,25,0,99.86,0.09,0.05
29,NE,4953,4946,3,4,4932,13,1,99.86,0.06,0.08
33,NV,11878,11860,13,5,11823,37,0,99.85,0.11,0.04
45,VA,25455,25414,31,10,25366,46,2,99.84,0.12,0.04
20,MD,18351,18320,27,4,18270,49,1,99.83,0.15,0.02
48,WI,13646,13623,20,3,13573,49,1,99.83,0.15,0.02


### Bottom 10 states by tract resolution rate (>= 1,000 rows)

,state_abbrev,total_rows,resolved_rows,ambiguous_rows,missing_rows,one_to_one_rows,tie_via_zip_rows,tie_unanimous_rows,resolution_rate_pct,ambiguous_rate_pct,missing_rate_pct
28,ND,1748,1730,18,0,1717,13,0,98.97,1.03,0.00
26,MT,1959,1941,16,2,1933,8,0,99.08,0.82,0.10
10,GA,36736,36434,253,49,36208,219,7,99.18,0.69,0.13
32,NM,5291,5260,23,8,5235,24,1,99.41,0.43,0.15
9,FL,79864,79414,320,130,78925,474,15,99.44,0.40,0.16
40,SC,19076,18974,86,16,18880,93,1,99.47,0.45,0.08
36,OK,12095,12034,24,37,11992,40,2,99.50,0.20,0.31
1,AL,11378,11321,47,10,11227,88,6,99.50,0.41,0.09
15,IN,17857,17775,71,11,17690,85,0,99.54,0.40,0.06
46,VT,1101,1096,5,0,1096,0,0,99.55,0.45,0.00


### Top 10 states by row count

,state_abbrev,total_rows,resolved_rows,ambiguous_rows,missing_rows,one_to_one_rows,tie_via_zip_rows,tie_unanimous_rows,resolution_rate_pct,ambiguous_rate_pct,missing_rate_pct
4,CA,165793,165498,271,24,165113,379,6,99.82,0.16,0.01
43,TX,88171,87826,268,77,87538,280,8,99.61,0.30,0.09
9,FL,79864,79414,320,130,78925,474,15,99.44,0.40,0.16
34,NY,62311,62184,99,28,61929,250,5,99.80,0.16,0.04
14,IL,39768,39635,102,31,39427,204,4,99.67,0.26,0.08
38,PA,37662,37547,78,37,37412,134,1,99.69,0.21,0.10
35,OH,37188,37086,81,21,36954,130,2,99.73,0.22,0.06
10,GA,36736,36434,253,49,36208,219,7,99.18,0.69,0.13
27,NC,34365,34251,102,12,34078,169,4,99.67,0.30,0.03
22,MI,31221,31161,49,11,31059,100,2,99.81,0.16,0.04
